# Data Prep

In [ ]:
import pandas as pd
df = pd.read_csv('train.csv')
calender_events =  pd.read_csv('calendar_events.csv')
df = df.merge( calender_events, on='date', how = 'left')
#df

,store_id,store_name,date,revenue,event
0,0,All Stores,2011-01-29,204126.52,NaN
1,0,All Stores,2011-01-30,197426.42,NaN
2,0,All Stores,2011-01-31,144267.27,NaN
3,0,All Stores,2011-02-01,151903.00,NaN
4,0,All Stores,2011-02-02,117399.88,NaN
...,...,...,...,...,...
18761,10,Wisconsin – Badger Crossing,2015-09-26,25689.55,NaN
18762,10,Wisconsin – Badger Crossing,2015-09-27,26557.53,NaN
18763,10,Wisconsin – Badger Crossing,2015-09-28,19067.53,NaN
18764,10,Wisconsin – Badger Crossing,2015-09-29,16467.95,NaN


In [ ]:
#make events a binary yes or no
df['event_binary'] = df['event'].notna().astype(int)
#make features lagged
df = df.sort_values(['store_id', 'date'])

for lag in [1, 2, 7, 30, 90]:
    df[f'lag_{lag}'] = df.groupby('store_id')['revenue'].shift(lag)


#features day of week and month 
df['date'] = pd.to_datetime(df['date'])
df['day_of_week'] = df['date'].dt.dayofweek
df['month'] = df['date'].dt.month


features = ['event_binary', 'lag_1', 'lag_2', 'day_of_week', 'month','lag_week' ,'lag_month', 'lag_quarter' ]


# #tomorrws revenue
# df['target_daily'] = df.groupby('store_id')['revenue'].shift(-1)
# #todays revenue 
# df['target_monthly'] = df.groupby('store_id')['revenue'].shift(-30)



,store_id,store_name,date,revenue_today,event,event_binary,lag_1,lag_2,lag_week,lag_month,lag_quarter,day_of_week,month,lag_7,lag_30,lag_90
0,0,All Stores,2011-01-29,204126.52,NaN,0,NaN,NaN,NaN,NaN,NaN,5,1,NaN,NaN,NaN
1,0,All Stores,2011-01-30,197426.42,NaN,0,204126.52,NaN,NaN,NaN,NaN,6,1,NaN,NaN,NaN
2,0,All Stores,2011-01-31,144267.27,NaN,0,197426.42,204126.52,NaN,NaN,NaN,0,1,NaN,NaN,NaN
3,0,All Stores,2011-02-01,151903.00,NaN,0,144267.27,197426.42,NaN,NaN,NaN,1,2,NaN,NaN,NaN
4,0,All Stores,2011-02-02,117399.88,NaN,0,151903.00,144267.27,NaN,NaN,NaN,2,2,NaN,NaN,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
18761,10,Wisconsin – Badger Crossing,2015-09-26,25689.55,NaN,0,21475.47,17646.33,28280.03,17735.05,23104.83,5,9,28280.03,17735.05,23104.83
18762,10,Wisconsin – Badger Crossing,2015-09-27,26557.53,NaN,0,25689.55,21475.47,28233.15,20053.50,18231.20,6,9,28233.15,20053.50,18231.20
18763,10,Wisconsin – Badger Crossing,2015-09-28,19067.53,NaN,0,26557.53,25689.55,19653.90,23304.33,17573.20,0,9,19653.90,23304.33,17573.20
18764,10,Wisconsin – Badger Crossing,2015-09-29,16467.95,NaN,0,19067.53,26557.53,17487.03,24047.72,20421.38,1,9,17487.03,24047.72,20421.38


In [20]:

def create_train_and_test_data(df):
    features = ['event_binary', 'lag_1', 'lag_2', 'day_of_week', 'month', 'lag_week', 'lag_month', 'lag_quarter']

    X = df[features + ['date'] + ['store_name']]
    y_day = df[['revenue', 'date']]


    #logic here is that current date is current revenue which is what being predicted so already 1 day ahead +29 = 30, etc.
    y_month = df[['date']].copy()
    y_month['revenue'] = df.groupby('store_id')['revenue'].shift(-29)

    y_quarter = df[['date']].copy()
    y_quarter['revenue'] = df.groupby('store_id')['revenue'].shift(-89)

    split_date = df['date'].quantile(0.8, interpolation='nearest')

    # Daily
    X_train_day = df[df['date'] <= split_date][features]
    y_train_day = y_day[df['date'] <= split_date]['revenue']
    X_test_day  = df[df['date'] > split_date][features]
    y_test_day  = y_day[df['date'] > split_date]['revenue']

    daily = (X_train_day,y_train_day,X_test_day, y_test_day )
    # Monthly — drop NaN rows caused by shift(-29)
    df_month = df.copy()
    df_month['target'] = y_month['revenue']
    df_month = df_month.dropna(subset=['target'])

    X_train_month = df_month[df_month['date'] <= split_date][features]
    y_train_month = df_month[df_month['date'] <= split_date]['target']
    X_test_month  = df_month[df_month['date'] > split_date][features]
    y_test_month  = df_month[df_month['date'] > split_date]['target']
    
    
    monthly = (X_train_month,y_train_month ,X_test_month,y_test_month )
    # Quarterly — drop NaN rows caused by shift(-89)
    df_quarter = df.copy()
    df_quarter['target'] = y_quarter['revenue']
    df_quarter = df_quarter.dropna(subset=['target'])

    X_train_quarter = df_quarter[df_quarter['date'] <= split_date][features]
    y_train_quarter = df_quarter[df_quarter['date'] <= split_date]['target']
    X_test_quarter  = df_quarter[df_quarter['date'] > split_date][features]
    y_test_quarter  = df_quarter[df_quarter['date'] > split_date]['target']
    
    quarterly = (X_train_quarter, y_train_quarter, X_test_quarter, y_test_quarter)
   
    return daily, monthly, quarterly 

# Creating Models /Training

In [30]:
from xgboost import XGBRegressor
def create_model(X_train, y_train):
    model = XGBRegressor(
        n_estimators=1000,
        learning_rate=0.05,
        max_depth=6,
        early_stopping_rounds=50,
        eval_metric='rmse'
    )
    
    model.fit(
        X_train, y_train,
        eval_set=[(X_train, y_train)],
        verbose=False
    )
    
    return model
    

for store in df['store_name'].unique():
    daily, monthly, quarterly = create_train_and_test_data(df[df['store_name'] == store])
    
    daily_model = create_model(daily[0], daily[1])
    monthly_model = create_model(monthly[0], monthly[1])
    quarterly_model = create_model(quarterly[0], quarterly[1])

    

KeyboardInterrupt: 

In [ ]:
daily_model

P